# Experiment Journal

**This is the only notebook to keep editing.** It is the live entry point for the experiment journal.

All other notebooks (`01_…` through `11_…`) were one-shot scaffolds; the audit at `papers/experiments/README.md` shows that fewer than half of them actually ran to completion. New experiments go on a card in [`papers/experiments/`](../papers/experiments/), not into a new notebook.

## Contents

1. [Project state at a glance](#state) — what works, what's broken, what's next
2. [Helpers](#helpers) — `log_experiment()`, `show_results()`, `summarise_tables()`
3. [Run a new analysis](#run) — template cell to drop in fresh code

## How to add an experiment

1. Create `papers/experiments/E##_<short_name>.md` from the template at the bottom of this notebook.
2. Add a row to the index in `papers/experiments/README.md`.
3. Use [the helpers below](#helpers) to log numbers to `papers/midl2026/RESULTS.tsv`.
4. Commit with a message of the form `E## <author tag>: <one-liner>`.

<a id='state'></a>
## Project state at a glance

Run the next cell to print the live status. Numbers come straight from the on-disk tables and `RESULTS.tsv`.

In [ ]:
from pathlib import Path
import re
import pandas as pd

REPO = Path('..').resolve() if Path('papers').exists() else Path('.').resolve()
TABLES = REPO / 'papers' / 'tables'
RESULTS = REPO / 'papers' / 'midl2026' / 'RESULTS.tsv'
EXP_DIR = REPO / 'papers' / 'experiments'

# 1. on-disk data tables (per-experiment outputs)
rows = []
for csv in sorted(TABLES.glob('*.csv')):
    df = pd.read_csv(csv, on_bad_lines='skip')
    rows.append({'file': csv.name, 'rows': len(df), 'cols': df.shape[1]})
print('Data tables in papers/tables/:')
print(pd.DataFrame(rows).to_string(index=False))

# 2. experiment cards
cards = sorted(EXP_DIR.glob('E*.md'))
print(f'\n{len(cards)} experiment cards in papers/experiments/')
for c in cards:
    title = next((l for l in c.read_text(encoding='utf-8').splitlines()
                  if l.startswith('# ')), '?').lstrip('# ')
    print(f'  {c.name:<42s} {title}')

# 3. cumulative results log
if RESULTS.exists():
    res = pd.read_csv(RESULTS, sep='\t', on_bad_lines='skip')
    print(f'\nRESULTS.tsv: {len(res)} entries across '
          f'{res["tag"].nunique()} tags')
    print(res.tail(8).to_string(index=False))
else:
    print('\nRESULTS.tsv not yet created.')

<a id='helpers'></a>
## Helpers

In [ ]:
import datetime as _dt


def log_experiment(tag: str, verdict: str = '', **kw):
    """Append a row to RESULTS.tsv. `tag` should match the experiment
    card ID (e.g. 'E11_partial_corr'). `verdict` is one of
    ACCEPT / REJECT / AMBIGUOUS / NA. Extra keyword arguments are written
    as columns; all metric values become floats with %.4g formatting."""
    RESULTS.parent.mkdir(parents=True, exist_ok=True)
    header_needed = not RESULTS.exists()
    ts = _dt.datetime.now().isoformat(timespec='seconds')
    cols = ['ts', 'tag', 'verdict'] + list(kw.keys())
    row = [ts, tag, verdict] + [
        f'{v:.4g}' if isinstance(v, float) else str(v) for v in kw.values()
    ]
    with RESULTS.open('a', encoding='utf-8') as f:
        if header_needed:
            f.write('\t'.join(cols) + '\n')
        f.write('\t'.join(row) + '\n')
    print(f'logged → RESULTS.tsv  {tag}  verdict={verdict}  '
          f'{"  ".join(f"{k}={v}" for k, v in kw.items())}')


def show_results(tag_prefix: str = ''):
    """Pretty-print RESULTS.tsv, optionally filtered by tag prefix."""
    if not RESULTS.exists():
        print('No RESULTS.tsv yet.')
        return
    res = pd.read_csv(RESULTS, sep='\t', on_bad_lines='skip')
    if tag_prefix:
        res = res[res['tag'].str.startswith(tag_prefix)]
    return res


def open_card(experiment_id: str):
    """Print an experiment card to the cell output."""
    matches = list(EXP_DIR.glob(f'{experiment_id}*.md'))
    if not matches:
        print(f'No card matching {experiment_id}.')
        return
    print(matches[0].read_text(encoding='utf-8'))


print('Helpers ready: log_experiment, show_results, open_card')

### Quick smoke test

View what's in `RESULTS.tsv` for the existing overnight ratchet experiments:

In [ ]:
show_results()

<a id='run'></a>
## Run a new analysis

Drop your code into the cell below. The pattern is:

1. Pre-register: write down the hypothesis and the decision rule **before** running.
2. Run the analysis. Keep it deterministic (set seeds).
3. Call `log_experiment(tag, verdict, **metrics)` with named metrics.
4. Write up the result in a new card under `papers/experiments/`.
5. Commit. Push to the appropriate feature branch.

### Template — copy and adapt

In [ ]:
# ── Template — replace with your actual analysis ─────────────────────
#
# import numpy as np, pandas as pd, scipy.stats as ss
#
# # 1) pre-registration (also written in the card before this code runs)
# #    Hypothesis: ...
# #    Decision rule: ACCEPT if metric < threshold, REJECT if metric > ...
#
# # 2) load data
# # df = pd.read_csv(TABLES / 'simon_brainage_synthba.csv')
#
# # 3) compute
# # ...
#
# # 4) decide & log
# verdict = 'ACCEPT' if cond else 'REJECT'
# log_experiment(
#     tag='E14_my_new_test',
#     verdict=verdict,
#     n=len(df),
#     statistic=float(value),
#     p_value=float(p),
# )
#
# # 5) write the card under papers/experiments/E14_my_new_test.md
# ─────────────────────────────────────────────────────────────────────
pass

## Card template

When you create `papers/experiments/E##_<short_name>.md`, copy this:

```markdown
# E## — Short title

- **Author**: KK / RK / GB / CL
- **Date**: YYYY-MM-DD
- **Status**: data-saved / done / ran-but-no-csv / abandoned
- **Branch / commit**: <hash>

## Question

## Method

## Result

| metric | value |
|---|---|

## Interpretation

## What's left undone

## Pointers
- code:
- data:

## Next
```

## Active hypotheses → next ratchet

Always check `papers/experiments/README.md → Active hypotheses` for the current queue. Pick one, do it, log, write the card, commit.

## Reference docs

- [`papers/related_works/sota_design.md`](../papers/related_works/sota_design.md) — SOTA experimental design 2024–2026
- [`papers/related_works/data_audit.md`](../papers/related_works/data_audit.md) — current cohort coverage
- [`papers/midl2026/HYPOTHESIS.md`](../papers/midl2026/HYPOTHESIS.md) — Karpathy-style pre-registration template (E11)
- [`papers/midl2026/FINDINGS.md`](../papers/midl2026/FINDINGS.md) — overturned-claim writeup (E11/E12)